In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter


from sklearn.cluster import KMeans
import json

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
root_dir = r'smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [ ]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"] 

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [ ]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':  
        return 0
    else:
        return 1

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

In [ ]:
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  

inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
df = df.dropna()
df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
labels = df['Anomaly_Label']
attack_types = df['Attack_Type'] 

feature_combos = [
   ['SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min'],
   ['SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
   ['Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
   ['SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Idle Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Flow IAT Max', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Max', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Packet Length Max', 'Dst Port_freq', 'Idle Mean', 'Packet Length Mean', 'Flow IAT Min'],
   ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean']
]

Elbow method to determine K:

In [ ]:
for idx, features in enumerate(feature_combos):
    print(f"\n=== Feature Combo {idx+1}: {features} ===")

    # Extract and scale features
    X = df[features].fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Elbow Method
    distortions = []
    K_range = range(1, 11)
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_scaled)
        distortions.append(kmeans.inertia_)

    plt.figure(figsize=(6, 4))
    plt.plot(K_range, distortions, 'bo-')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Distortion (Inertia)')
    plt.title(f'Elbow Method - Combo {idx+1}')
    plt.grid(True)
    plt.show()

In [ ]:
k_values = [
    [3,4],
    [5],
    [4],
    [5,6],
    [6],
    [4,5],
    [4,5,6],
    [5],
    [4],
    [5],
    [5,6],
    [5],
    [6,7],
    [6],
    [6],
    [6,7,8],
    [6,7,8],
    [7,8],
    [7,8],
    [7,8],
    [8],
    [7],
    [6,7,8],
    [6,7,8]
]

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X = df[feature_combos[0]].fillna(0)
X_scaled = StandardScaler().fit_transform(X)
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='coolwarm', alpha=0.3) #0 blue, 1 red
plt.title("True Labels in PCA Space")
plt.show()


k-means for the upper features and k values: 

In [ ]:
output_file = "kmeans_k=2_results.jsonl"
with open(output_file, 'w') as f_out:
    for i, features in enumerate(feature_combos):
        X = df[features].fillna(0)
        X_scaled = StandardScaler().fit_transform(X)

        for k in k_values[i]: 

            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            pred_clusters = kmeans.fit_predict(X_scaled)

            unique_clusters = np.unique(pred_clusters)
            print(f"Clusters formed: {unique_clusters}")

            pred_clusters = kmeans.labels_

            
            df_clusters = pd.DataFrame({
                "cluster": pred_clusters,
                "label":   labels  
            })
            counts = (
                df_clusters
                .groupby(["cluster", "label"])
                .size()
                .unstack(fill_value=0)
            )
            percentages = counts.div(counts.sum(axis=1), axis=0) * 100

            
            print(f"\n=== Feature combo #{i}, k={k} ===")
            print("Counts per cluster:")
            print(counts)
            print("\nPercentages per cluster:")
            print(percentages.round(2))

          
            X_pca = PCA(n_components=2).fit_transform(X_scaled)
            plt.scatter(X_pca[:, 0], X_pca[:, 1],
                        c=pred_clusters, cmap='viridis', alpha=0.3)
            plt.title(f"KMeans (k={k}) PCA – combo #{i}")
            plt.xlabel("PC1"); plt.ylabel("PC2")
            plt.show()


            
            cluster_label_map = {}
            
            for cluster in range(k):
                indices = np.where(pred_clusters == cluster)[0]
                if len(indices) > 0:
                    majority_label = np.argmax(np.bincount(labels.to_numpy()[indices]))

                    cluster_label_map[cluster] = majority_label
                else:
                    cluster_label_map[cluster] = 0

            pred_labels = np.array([cluster_label_map[c] for c in pred_clusters])
            cm = confusion_matrix(labels, pred_labels, labels=[0, 1])
            TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

            precision = TP / (TP + FP) if TP + FP > 0 else 0.0
            recall = TP / (TP + FN) if TP + FN > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
            total_anomaly_detection_rate = (TP / (TP + FN)) * 100 if TP + FN > 0 else 0.0

            

            attack_counts = {} 
            attack_hits = {} 

            for true, pred, attack in zip(labels, pred_labels, attack_types):
                if true == 1:
                    attack_counts[attack] = attack_counts.get(attack, 0) + 1
                    if pred == 1:
                        attack_hits[attack] = attack_hits.get(attack, 0) + 1 

            attack_percentages = {
                atk: round(100 * attack_hits.get(atk, 0) / total, 2)
                for atk, total in attack_counts.items()
            }

            json_line = {
                "feature_combo": features,
                "k": k,
                "confusion_matrix": cm.tolist(),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1_score": round(f1, 4),
                "total_anomaly_detection_rate": round(total_anomaly_detection_rate, 2),
                "attack_detection_percentages": attack_percentages
            }

            f_out.write(json.dumps(json_line) + "\n")

print(f"✅ Results saved to: {output_file}")